### Source Tables:
- `_exponent._bronze_epic_clarity.patient_3` — OCCUPATION (43 records populated)
- `_exponent._bronze_epic_clarity.pat_enc_2` — SMOKING_STATUS_C (currently unpopulated)

### Strategy:
- Extract occupation from patient_3 as observation (concept_id = 4135376, 'Occupation')
- Smoking status section included but commented out (zero data currently)
- observation_type_concept_id = 32817 (EHR)
- observation_date defaults to NULL (patient-level attribute, no specific date)

### Notes:
- This notebook depends on source_to_person being populated for epic_clarity
- Data is very sparse — only 43 occupation records out of 10M patients
- Gender identity, tobacco use, and ambulatory status columns are either absent or unpopulated
- Notebook structure is ready for additional observation sources as data becomes available

# Transformation

In [0]:
%sql
-- Create silver_observation temp view for Epic Clarity
CREATE OR REPLACE TEMPORARY VIEW silver_observation AS

-- Occupation from patient_3
SELECT
  4135376 AS observation_concept_id,  -- SNOMED: Occupation
  NULL AS observation_date,
  NULL AS observation_datetime,
  32817 AS observation_type_concept_id,  -- EHR
  NULL AS value_as_number,
  p3.OCCUPATION AS value_as_string,
  0 AS value_as_concept_id,
  0 AS qualifier_concept_id,
  0 AS unit_concept_id,
  NULL AS unit_source_value,
  NULL AS qualifier_source_value,
  p3.OCCUPATION AS value_source_value,
  'Occupation' AS observation_source_value,
  0 AS observation_source_concept_id,
  CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', p3.PAT_ID) AS person_source_value,
  NULL AS provider_source_value,
  NULL AS visit_occurrence_source_value,
  NULL AS visit_detail_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'patient_3', 'OCCUPATION', p3.PAT_ID) AS observation_source_key,
  'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.patient_3 p3
WHERE p3.OCCUPATION IS NOT NULL
  AND TRIM(p3.OCCUPATION) != ''


In [0]:
# %sql
# -- Preview
# SELECT * FROM silver_observation LIMIT 10

In [0]:
# %sql
# DESCRIBE _exponent.omop_silver.observation

In [0]:
%sql
ALTER TABLE _exponent.omop_silver.observation ALTER COLUMN person_id DROP NOT NULL

In [0]:
%sql
ALTER TABLE _exponent.omop_silver.observation ALTER COLUMN observation_type_concept_id DROP NOT NULL

In [0]:
%sql
ALTER TABLE _exponent.omop_silver.observation ALTER COLUMN observation_date DROP NOT NULL

# Write to Silver

In [0]:
%sql
MERGE INTO _exponent.omop_silver.observation AS t
USING silver_observation AS s
ON t.observation_source_value = s.observation_source_key

WHEN MATCHED AND (
     NOT (t.observation_concept_id <=> s.observation_concept_id)
  OR NOT (t.observation_date <=> s.observation_date)
  OR NOT (t.observation_type_concept_id <=> s.observation_type_concept_id)
  OR NOT (t.value_as_string <=> s.value_as_string)
  OR NOT (t.value_as_concept_id <=> s.value_as_concept_id)
  OR NOT (t.value_source_value <=> s.value_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.observation_concept_id        = s.observation_concept_id,
  t.observation_date              = s.observation_date,
  t.observation_datetime          = s.observation_datetime,
  t.observation_type_concept_id   = s.observation_type_concept_id,
  t.value_as_number               = s.value_as_number,
  t.value_as_string               = s.value_as_string,
  t.value_as_concept_id           = s.value_as_concept_id,
  t.qualifier_concept_id          = s.qualifier_concept_id,
  t.unit_concept_id               = s.unit_concept_id,
  t.unit_source_value             = s.unit_source_value,
  t.qualifier_source_value        = s.qualifier_source_value,
  t.value_source_value            = s.value_source_value,
  t.observation_source_value      = s.observation_source_value,
  t.observation_source_concept_id = s.observation_source_concept_id,
  t.person_source_value           = s.person_source_value,
  t.source_system                 = s.source_system

WHEN NOT MATCHED THEN INSERT (
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_source_value,
  observation_source_concept_id,
  person_source_value,
  source_system
)
VALUES (
  s.observation_concept_id,
  s.observation_date,
  s.observation_datetime,
  s.observation_type_concept_id,
  s.value_as_number,
  s.value_as_string,
  s.value_as_concept_id,
  s.qualifier_concept_id,
  s.unit_concept_id,
  s.unit_source_value,
  s.qualifier_source_value,
  s.value_source_value,
  s.observation_source_value,
  s.observation_source_concept_id,
  s.person_source_value,
  s.source_system
);

In [0]:
# %sql
# ALTER TABLE _exponent.omop_silver.observation ADD COLUMN person_source_value STRING

In [0]:
# %sql
# DELETE FROM _exponent.omop_silver.observation WHERE source_system = 'epic_clarity'

In [0]:
# %sql
# -- Verify silver
# SELECT * FROM _exponent.omop_silver.observation
# WHERE source_system = 'epic_clarity'
# LIMIT 10

# Register Observation IDs

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_observation (
    source_system,
    observation_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.observation_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, observation_source_value
    FROM _exponent.omop_silver.observation
    WHERE source_system = 'epic_clarity'
      AND observation_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation x
  ON s.observation_source_value = x.observation_source_value;

# Write to Gold

In [0]:
%sql
-- MERGE INTO _exponent.omop.observation AS gold
MERGE INTO _exponent.omop_epic.observation AS gold
USING (
  SELECT
    sob.observation_id,
    stp.person_id,
    s.observation_concept_id,
    s.observation_date,
    s.observation_datetime,
    s.observation_type_concept_id,
    s.value_as_number,
    s.value_as_string,
    s.value_as_concept_id,
    s.qualifier_concept_id,
    s.unit_concept_id,
    NULL AS provider_id,
    NULL AS visit_occurrence_id,
    NULL AS visit_detail_id,
    s.observation_source_value,
    s.observation_source_concept_id,
    s.unit_source_value,
    s.qualifier_source_value,
    s.value_source_value,
    NULL AS observation_event_id,
    0 AS obs_event_field_concept_id
  FROM _exponent.omop_silver.observation s
  JOIN _exponent.omop_mapping.source_to_observation sob
    ON sob.observation_source_value = s.observation_source_value
   AND sob.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
  WHERE s.source_system = 'epic_clarity'
) AS src
ON gold.observation_id = src.observation_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                      = src.person_id,
  gold.observation_concept_id         = src.observation_concept_id,
  gold.observation_date               = src.observation_date,
  gold.observation_datetime           = src.observation_datetime,
  gold.observation_type_concept_id    = src.observation_type_concept_id,
  gold.value_as_number                = src.value_as_number,
  gold.value_as_string                = src.value_as_string,
  gold.value_as_concept_id            = src.value_as_concept_id,
  gold.qualifier_concept_id           = src.qualifier_concept_id,
  gold.unit_concept_id                = src.unit_concept_id,
  gold.provider_id                    = src.provider_id,
  gold.visit_occurrence_id            = src.visit_occurrence_id,
  gold.visit_detail_id                = src.visit_detail_id,
  gold.observation_source_value       = src.observation_source_value,
  gold.observation_source_concept_id  = src.observation_source_concept_id,
  gold.unit_source_value              = src.unit_source_value,
  gold.qualifier_source_value         = src.qualifier_source_value,
  gold.value_source_value             = src.value_source_value

WHEN NOT MATCHED THEN INSERT (
  observation_id,
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id
)
VALUES (
  src.observation_id,
  src.person_id,
  src.observation_concept_id,
  src.observation_date,
  src.observation_datetime,
  src.observation_type_concept_id,
  src.value_as_number,
  src.value_as_string,
  src.value_as_concept_id,
  src.qualifier_concept_id,
  src.unit_concept_id,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.observation_source_value,
  src.observation_source_concept_id,
  src.unit_source_value,
  src.qualifier_source_value,
  src.value_source_value,
  src.observation_event_id,
  src.obs_event_field_concept_id
);

# Validation

In [0]:
# %sql
# -- Layer counts
# SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.observation WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_observation WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.observation

In [0]:
# %sql
# -- Verify gold
# SELECT * FROM _exponent.omop.observation
# LIMIT 10